# 🧩 Mini-Lab: First Win — Triage Your Own Text

**Module 1: Ship a Working AI Feature Today** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Run** a triage on your own text with no install, no account, and no API key
2. **Read** a structured JSON result with `id`, `label`, `summary`, and `latency_ms`
3. **Recognize** that a plausible mock label is not always a correct label
4. **Map** the cells you just ran onto the six-box pipeline (ingest, prepare, call, validate, log, evaluate)

## Target Concepts

| Concept | Gloss |
|---------|-------|
| AI Pipeline Stages | A practical AI feature pipeline with six boxes: ingest, prepare, call, validate, log, evaluate. The model call is just one box, not the whole system. |


## Why This Matters

Mira is handed 4,000 unread support messages on a Friday. Leadership wants them sorted by morning: billing, bugs, account issues, feature requests, everything else.

You will not solve 4,000 rows today. You will do something better: **drive the whole path from "text goes in" to "label comes out" on a message you choose**, in the browser, with nothing to install.

**Privacy:** paste sample text, or personal-but-harmless text only. Never confidential or customer data — even into this local mock. The mock does not leave your session, but the habit starts now.

If this notebook will not open, download the `.ipynb` and run it in any Python 3 Jupyter environment. There is no account and no key.

Run the cells **top to bottom**. A later cell uses names defined above it.

In [22]:
import json
import time

from IPython.display import display, Markdown


def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## A Stand-In for a Model Call

`classify_message` looks like an AI call: text in, structured result out. It is not a model. It is a **mock** — a keyword function that returns a plausible label so you can run the rest of the work with zero setup.

Allowed labels: `billing`, `bug`, `account`, `feature`, `other`.

In [23]:
LABELS = ("billing", "bug", "account", "feature", "other")

KEYWORD_RULES = (
    ("billing", ("invoice", "refund", "charged", "payment", "receipt", "subscription")),
    ("bug", ("crash", "error", "broken", "bug", "not working")),
    ("account", ("password", "login", "sign in", "locked out", "username")),
    ("feature", ("would be nice", "feature", "can you add", "wishlist")),
)


def mock_infer(text):
    """Return a plausible label and a short summary. Not always correct."""
    lowered = text.lower()
    label = "other"
    for name, keywords in KEYWORD_RULES:
        if any(keyword in lowered for keyword in keywords):
            label = name
            break
    summary = " ".join(text.split())
    if len(summary) > 100:
        summary = summary[:97] + "..."
    return {"label": label, "summary": summary}


def classify_message(text, msg_id="local-1"):
    """Take one message, call the mock, check the keys, return a JSON-ready dict."""
    started = time.perf_counter()
    cleaned = (text or "").strip()
    raw = mock_infer(cleaned)
    result = {
        "id": msg_id,
        "label": raw["label"],
        "summary": raw["summary"],
        "latency_ms": round((time.perf_counter() - started) * 1000, 2),
    }
    required = ("id", "label", "summary", "latency_ms")
    missing = [key for key in required if key not in result]
    if missing:
        raise ValueError(f"result is missing keys: {missing}")
    if result["label"] not in LABELS:
        raise ValueError(f"label {result['label']!r} is not in {LABELS}")
    return result


print("classify_message is ready. Allowed labels:", ", ".join(LABELS))

classify_message is ready. Allowed labels: billing, bug, account, feature, other


## Your First Win

The next cell is the whole move: **paste a message, run one cell, read JSON**.

Keep the sample, or replace `YOUR_TEXT` with your own non-confidential message. Then run the cell.

In [24]:
YOUR_TEXT = "I was charged twice for invoice 1842. Please refund the extra payment."

result = classify_message(YOUR_TEXT, msg_id="demo-001")
print(json.dumps(result, indent=2))

assert set(result) >= {"id", "label", "summary", "latency_ms"}
print("Shape check passed.")

{
  "id": "demo-001",
  "label": "billing",
  "summary": "I was charged twice for invoice 1842. Please refund the extra payment.",
  "latency_ms": 0.01
}
Shape check passed.


## Read the Four Fields

That printout is not a chat reply. It is a **record** — something you can store, count, and check.

In [25]:
md(
    f"| Field | Value | What it is |\n"
    f"|-------|-------|------------|\n"
    f"| `id` | `{result['id']}` | Which message this row is about |\n"
    f"| `label` | `{result['label']}` | The bucket the mock chose |\n"
    f"| `summary` | {result['summary']} | A short copy of the input |\n"
    f"| `latency_ms` | {result['latency_ms']} | How long the call took, in milliseconds |\n"
)

| Field | Value | What it is |
|-------|-------|------------|
| `id` | `demo-001` | Which message this row is about |
| `label` | `billing` | The bucket the mock chose |
| `summary` | I was charged twice for invoice 1842. Please refund the extra payment. | A short copy of the input |
| `latency_ms` | 0.01 | How long the call took, in milliseconds |


## The Mock Is Not Magic

Keyword rules pick the **first** matching bucket. That looks confident. It is still a guess.

The next cell is a real support-shaped message. Read it yourself before you run. What label would **you** give it?

In [26]:
tricky = "The refund button on the login page is broken."

tricky_result = classify_message(tricky, msg_id="demo-002")
print(json.dumps(tricky_result, indent=2))
print()
print("A human would likely call this a bug: a broken button.")
print("The mock saw 'refund' first and called it billing.")
print("A plausible label is not the same as a correct label.")

{
  "id": "demo-002",
  "label": "billing",
  "summary": "The refund button on the login page is broken.",
  "latency_ms": 0.01
}

A human would likely call this a bug: a broken button.
The mock saw 'refund' first and called it billing.
A plausible label is not the same as a correct label.


---
## Map the six boxes

The first win is not 'a model demo'. Name the stage each cell just occupied.


## The Six-Box AI Pipeline

Every AI feature you ship — a chatbot, a classifier, a document extractor — runs through the same six boxes. The model call is only one of them:

<div style='display:flex;flex-wrap:wrap;align-items:center;gap:8px;margin:12px 0 16px 0;'>
<span style='border:1px solid #888;border-radius:6px;padding:8px 12px;text-align:center;'><strong>1. Ingest</strong></span>
<span>→</span>
<span style='border:1px solid #888;border-radius:6px;padding:8px 12px;text-align:center;'><strong>2. Prepare</strong></span>
<span>→</span>
<span style='border:1px solid #888;border-radius:6px;padding:8px 12px;text-align:center;'><strong>3. Call</strong></span>
<span>→</span>
<span style='border:1px solid #888;border-radius:6px;padding:8px 12px;text-align:center;'><strong>4. Validate</strong></span>
<span>→</span>
<span style='border:1px solid #888;border-radius:6px;padding:8px 12px;text-align:center;'><strong>5. Log</strong></span>
<span>→</span>
<span style='border:1px solid #888;border-radius:6px;padding:8px 12px;text-align:center;'><strong>6. Evaluate</strong></span>
</div>

You have already run all six. Here is where each box lives in the cells above:

| # | Box | What happens | In the cells you just ran |
|---|-----|--------------|---------------------------|
| 1 | **Ingest** | Bring raw input in (file, API, user text) | `YOUR_TEXT` — the message you pasted |
| 2 | **Prepare** | Clean and shape the input for the model | `(text or "").strip()` inside `classify_message` |
| 3 | **Call** | Send the prepared input to the model | `mock_infer(cleaned)` — a stand-in for a real model call |
| 4 | **Validate** | Check the result has the promised shape and allowed values | The `required` keys check and the `LABELS` check |
| 5 | **Log** | Record what happened so you can inspect it later | `id` and `latency_ms` on every result |
| 6 | **Evaluate** | Judge the output against what a person would say | The tricky message: the mock said `billing`, a person says `bug` |

## Define the Pipeline Stages

Let's encode the six stages so we can reference them programmatically.

> 🧩 **AI Pipeline Stages.** The six boxes from the video, now written as code.

In [27]:
PIPELINE_STAGES = {
    1: {"name": "Ingest",   "does": "Bring raw input in from a file, an API, or a user"},
    2: {"name": "Prepare",  "does": "Clean and shape the input for the model, including the prompt"},
    3: {"name": "Call",     "does": "Send the prepared input to the model and get a raw result"},
    4: {"name": "Validate", "does": "Check the result has the promised shape and allowed values"},
    5: {"name": "Log",      "does": "Record what went in, what came out, and how long it took"},
    6: {"name": "Evaluate", "does": "Judge the outputs against what a person would say"},
}

# Print the pipeline
header = "| # | Stage | What It Does |\n|---|-------|-------------|\n"
rows = ""
for num, info in PIPELINE_STAGES.items():
    rows += f"| {num} | **{info['name']}** | {info['does']} |\n"

md(header + rows)

| # | Stage | What It Does |
|---|-------|-------------|
| 1 | **Ingest** | Bring raw input in from a file, an API, or a user |
| 2 | **Prepare** | Clean and shape the input for the model, including the prompt |
| 3 | **Call** | Send the prepared input to the model and get a raw result |
| 4 | **Validate** | Check the result has the promised shape and allowed values |
| 5 | **Log** | Record what went in, what came out, and how long it took |
| 6 | **Evaluate** | Judge the outputs against what a person would say |


## Locate Failures in the Pipeline

The real skill isn't memorizing the six stages — it's **looking at an error and immediately knowing which box it belongs to**. Let's practice with realistic failure scenarios.

In [28]:
# Realistic failure scenarios an AI engineer encounters
failures = [
    {
        "description": "FileNotFoundError: [Errno 2] No such file or directory: 'data/reviews.csv'",
        "stage": 1,
        "explanation": "The data file couldn't be loaded — this is an Ingest failure."
    },
    {
        "description": "KeyError: 'customer_name' when building the prompt template.",
        "stage": 2,
        "explanation": "The input wasn't shaped into a prompt correctly — this is a Prepare failure."
    },
    {
        "description": "openai.AuthenticationError: Invalid API key provided.",
        "stage": 3,
        "explanation": "The model call itself failed — this is a Call failure."
    },
    {
        "description": "json.decoder.JSONDecodeError: Expecting ',' — the model returned markdown instead of JSON.",
        "stage": 4,
        "explanation": "The result doesn't have the promised shape — this is a Validate failure."
    },
    {
        "description": "A customer disputes yesterday's label, but nothing recorded which message produced it.",
        "stage": 5,
        "explanation": "There is no record to inspect — this is a Log failure."
    },
    {
        "description": "The team shipped after three outputs 'looked good'; on real messages a third of the labels are wrong.",
        "stage": 6,
        "explanation": "Nobody checked outputs against what a person would say — this is an Evaluate failure."
    },
]

In [29]:
def locate_failure(failure):
    """Given a failure dict, return a formatted diagnosis."""
    stage_num = failure["stage"]
    stage_name = PIPELINE_STAGES[stage_num]["name"]
    return (
        f"**Failure:** {failure['description']}\n\n"
        f"📍 **Stage {stage_num} — {stage_name}**\n\n"
        f"*Why:* {failure['explanation']}\n\n---"
    )

In [30]:
# Walk through each failure and locate it in the pipeline
output = "## Failure → Stage Mapping\n\n"
for f in failures:
    output += locate_failure(f) + "\n"

md(output)

## Failure → Stage Mapping

**Failure:** FileNotFoundError: [Errno 2] No such file or directory: 'data/reviews.csv'

📍 **Stage 1 — Ingest**

*Why:* The data file couldn't be loaded — this is an Ingest failure.

---
**Failure:** KeyError: 'customer_name' when building the prompt template.

📍 **Stage 2 — Prepare**

*Why:* The input wasn't shaped into a prompt correctly — this is a Prepare failure.

---
**Failure:** openai.AuthenticationError: Invalid API key provided.

📍 **Stage 3 — Call**

*Why:* The model call itself failed — this is a Call failure.

---
**Failure:** json.decoder.JSONDecodeError: Expecting ',' — the model returned markdown instead of JSON.

📍 **Stage 4 — Validate**

*Why:* The result doesn't have the promised shape — this is a Validate failure.

---
**Failure:** A customer disputes yesterday's label, but nothing recorded which message produced it.

📍 **Stage 5 — Log**

*Why:* There is no record to inspect — this is a Log failure.

---
**Failure:** The team shipped after three outputs 'looked good'; on real messages a third of the labels are wrong.

📍 **Stage 6 — Evaluate**

*Why:* Nobody checked outputs against what a person would say — this is an Evaluate failure.

---


In [31]:
# Try your own: which stage?
my_error = "RateLimitError: You exceeded your current quota."
print(f"'{my_error}' → Stage 3: Call")

'RateLimitError: You exceeded your current quota.' → Stage 3: Call


## Quick Self-Check

Let's verify you can map stages by name. The function below takes a stage name and returns its number and description.

In [32]:
def lookup_stage(name):
    """Look up a pipeline stage by name (case-insensitive)."""
    for num, info in PIPELINE_STAGES.items():
        if info["name"].lower() == name.lower():
            return f"Stage {num}: **{info['name']}** — {info['does']}"
    return f"Unknown stage: '{name}'. Valid stages: {[s['name'] for s in PIPELINE_STAGES.values()]}"

# Test a few lookups
for stage_name in ["Ingest", "Validate", "Evaluate"]:
    md(lookup_stage(stage_name))

Stage 1: **Ingest** — Bring raw input in from a file, an API, or a user

Stage 4: **Validate** — Check the result has the promised shape and allowed values

Stage 6: **Evaluate** — Judge the outputs against what a person would say

## 🎯 Summary

### Key Takeaways

1. **Text in, structured JSON out** — a usable AI feature returns a record (`id`, `label`, `summary`, `latency_ms`), not a chat paragraph
2. **A mock is enough to start** — you can run the path today with no install, no account, and no API key
3. **Plausible is not correct** — a confident JSON label can still be the wrong bucket; the mock is a stand-in, not a judge of quality
4. **Six boxes** — ingest, prepare, call, validate, log, evaluate; the model call is one box, and every failure lives in one of them